# Auto recipe store example

Shows the read-only auto store. It exposes registered fixes as one-step recipes.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.stores import AutoRecipeStore
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset where `tas` uses Celsius units.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

The auto store lists registered fixes as generated recipe ids.

In [ ]:
store = AutoRecipeStore()

[recipe.id for recipe in store.list_recipes() if recipe.id.startswith("woodpecker.")]

Lookup returns generated recipes whose fix `matches()` method applies to the dataset.

In [ ]:
matched_plans = store.lookup(dataset)
[recipe.id for recipe in matched_plans]

Use the generated recipe through the public recipe API.

In [ ]:
recipe_id = "woodpecker.normalize_tas_units_to_kelvin"

findings = woodpecker.recipe.check(dataset, woodpecker.recipe.auto(recipe_id))
findings.fix_ids

In [ ]:
write = woodpecker.recipe.fix(dataset, woodpecker.recipe.auto(recipe_id), dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)